<a href="https://colab.research.google.com/github/MacKumachin/Clifford-FBSM-SignalControl/blob/main/Capital_Accumulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import matplotlib.pyplot as plt
import arviz as az

# ==========================================
# 1. データの準備 (モックデータの生成)
# ※実際には truss_fx_merged.csv 等を読み込み、
#   標準化 (Eq. 37) した配列をここに代入してください。
# ==========================================
np.random.seed(42)
T = 25  # ウィンドウの日数 (例: t = -5 から +19)
W_rupture_start = 8  # t = +3 に相当するインデックス
W_rupture_end = 10   # t = +5 に相当するインデックス

# 観測データ (標準化済みを想定)
z_gilt = np.random.normal(0, 1, T)
z_fx = np.random.normal(0, 1, T)
z_liq = np.random.normal(0, 1, T)
z_reset = np.random.normal(0, 1, T)

# イベントダミー
d_MB = np.zeros(T); d_MB[5:8] = 1   # Mini-budget shock
d_BoE = np.zeros(T); d_BoE[8:11] = 1 # BoE intervention
d_REV = np.zeros(T); d_REV[11:] = 1  # Policy reversal

# ==========================================
# 2. PyMCによる状態空間モデルの構築
# ==========================================
with pm.Model() as truss_model:

    # --- パラメータの事前分布 ---
    # 観測方程式の切片
    alpha = pm.Normal('alpha', mu=0, sigma=1, shape=4)

    # 観測方程式の係数 (すべて正の制約: Eq. 21)
    beta_11 = pm.HalfNormal('beta_11', sigma=1)
    beta_12 = pm.HalfNormal('beta_12', sigma=1)
    beta_13 = pm.HalfNormal('beta_13', sigma=1)
    beta_21 = pm.HalfNormal('beta_21', sigma=1)
    beta_22 = pm.HalfNormal('beta_22', sigma=1)
    beta_23 = pm.HalfNormal('beta_23', sigma=1)
    beta_31 = pm.HalfNormal('beta_31', sigma=1)
    beta_32 = pm.HalfNormal('beta_32', sigma=1)
    beta_41 = pm.Normal('beta_41', mu=0, sigma=1) # ダミーの係数は任意
    beta_42 = pm.Normal('beta_42', mu=0, sigma=1)

    # 観測ノイズの標準偏差
    sigma_obs = pm.HalfNormal('sigma_obs', sigma=1, shape=4)

    # AR(1)の自己回帰係数 (定常性を仮定)
    rhos = pm.Uniform('rhos', lower=-0.99, upper=0.99, shape=5)
    sigma_states = pm.HalfNormal('sigma_states', sigma=0.5, shape=5)

    # --- 潜在状態のダイナミクス (Eq. 24) ---
    # PyMCの機能を利用し、ダミー変数の影響(B*d_t)を含めたAR(1)過程を構築
    # ※実装の簡略化のため、状態変数(対数)をガウシアンランダムウォーク/ARで近似モデリング
    r_raw = pm.AR('r_raw', rho=rhos[0], sigma=sigma_states[0], shape=T)
    log_pi = pm.AR('log_pi', rho=rhos[1], sigma=sigma_states[1], shape=T)
    log_g = pm.AR('log_g', rho=rhos[2], sigma=sigma_states[2], shape=T)
    tilde_eta = pm.AR('tilde_eta', rho=rhos[3], sigma=sigma_states[3], shape=T)
    log_tau = pm.AR('log_tau', rho=rhos[4], sigma=sigma_states[4], shape=T)

    # 変数変換 (正値の保証: Eq. 35 等)
    r_t = pm.Deterministic('r_t', r_raw)
    pi_t = pm.Deterministic('pi_t', pt.exp(log_pi))
    g_t = pm.Deterministic('g_t', pt.exp(log_g))
    eta_t = pm.Deterministic('eta_t', pt.exp(tilde_eta))
    tau_t = pm.Deterministic('tau_t', pt.exp(log_tau))

    # --- 観測方程式 (Eq. 21) ---
    mu_gilt = alpha[0] + beta_11 * pi_t + beta_12 * eta_t + beta_13 * tau_t
    mu_fx = alpha[1] + beta_21 * pi_t - beta_22 * g_t + beta_23 * tau_t
    mu_liq = alpha[2] + beta_31 * eta_t + beta_32 * tau_t
    mu_reset = alpha[3] + beta_41 * d_BoE + beta_42 * d_REV

    pm.Normal('obs_gilt', mu=mu_gilt, sigma=sigma_obs[0], observed=z_gilt)
    pm.Normal('obs_fx', mu=mu_fx, sigma=sigma_obs[1], observed=z_fx)
    pm.Normal('obs_liq', mu=mu_liq, sigma=sigma_obs[2], observed=z_liq)
    pm.Normal('obs_reset', mu=mu_reset, sigma=sigma_obs[3], observed=z_reset)

    # --- 解析対象の事後指標 (Eq. 25, 26, 30) ---
    delta_t = pm.Deterministic('delta_t', pi_t + eta_t * (pi_t**2) - g_t)
    f1_t = pm.Deterministic('f1_t', r_t - pi_t)
    f2_t = pm.Deterministic('f2_t', g_t - r_t - eta_t * (r_t**2))
    f3_t = pm.Deterministic('f3_t', r_t + pi_t)
    C_t = pm.Deterministic('C_t', f2_t + f1_t + eta_t * f1_t * f3_t) # -delta_t と一致

    # --- MCMCサンプリング ---
    # 実環境では draws=2000, tune=1000 等に増やしてください
    trace = pm.sample(draws=1000, tune=1000, target_accept=0.9, chains=2, random_seed=42)

# ==========================================
# 3. 事後確率の計算と判定ルールの適用 (Appendix A.2 & A.3)
# ==========================================
# MCMCサンプルからの抽出
delta_samples = trace.posterior['delta_t'].stack(sample=("chain", "draw")).values
f1_samples = trace.posterior['f1_t'].stack(sample=("chain", "draw")).values
f2_samples = trace.posterior['f2_t'].stack(sample=("chain", "draw")).values
C_samples = trace.posterior['C_t'].stack(sample=("chain", "draw")).values

# 各時点での確率を計算 (Eq. 41 - 44)
prob_delta_pos = np.mean(delta_samples > 0, axis=1)
prob_C_neg = np.mean(C_samples < 0, axis=1)
prob_F = np.mean((f1_samples > 0) & (f2_samples > 0), axis=1)

# Rupture window における連続 m_* 日の条件チェック
m_star = 3
pass_flag = False

rupture_prob_delta = prob_delta_pos[W_rupture_start:W_rupture_end+1]
rupture_prob_F = prob_F[W_rupture_start:W_rupture_end+1]
rupture_prob_C = prob_C_neg[W_rupture_start:W_rupture_end+1]

# ウィンドウ内で連続して条件を満たす区間があるか探査
for i in range(len(rupture_prob_delta) - m_star + 1):
    cond_delta = np.all(rupture_prob_delta[i:i+m_star] >= 0.95)
    cond_F = np.all(rupture_prob_F[i:i+m_star] <= 0.05)
    cond_C = np.all(rupture_prob_C[i:i+m_star] >= 0.95)
    if cond_delta and cond_F and cond_C:
        pass_flag = True
        break

# 代表値として Rupture window 内の最大/最小確率等を取得（表出力用）
repr_prob_delta = np.max(rupture_prob_delta)
repr_prob_F = np.min(rupture_prob_F)
repr_prob_C = np.max(rupture_prob_C)
status = "Pass" if pass_flag else "Fail"

# ==========================================
# 4. Table 5 (Appendix Table A.1) 用の LaTeX行出力
# ==========================================
latex_row = f"Baseline window, baseline proxy, AR state & {repr_prob_delta:.3f} & {repr_prob_F:.3f} & {repr_prob_C:.3f} & {status} \\\\"

print("\n=== Table 5 (Appendix Table A.1) Row Output ===")
print(latex_row)
print("===============================================")

/usr/local/lib/python3.12/dist-packages/pymc/distributions/timeseries.py:595: UserWarning: Initial distribution not specified, defaulting to `Normal.dist(0, 100, shape=...)`. You can specify an init_dist manually to suppress this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pymc/distributions/timeseries.py:595: UserWarning: Initial distribution not specified, defaulting to `Normal.dist(0, 100, shape=...)`. You can specify an init_dist manually to suppress this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pymc/distributions/timeseries.py:595: UserWarning: Initial distribution not specified, defaulting to `Normal.dist(0, 100, shape=...)`. You can specify an init_dist manually to suppress this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pymc/distributions/timeseries.py:595: UserWarning: Initial distribution not specified, defaulting to `Normal.dist(0, 100, shape=...)`. You can specify an init_dist manually to suppress this wa

Output()

ERROR:pymc.stats.convergence:There were 222 divergences after tuning. Increase `target_accept` or reparameterize.
ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



=== Table 5 (Appendix Table A.1) Row Output ===
Baseline window, baseline proxy, AR state & 0.922 & 0.001 & 0.922 & Fail \\


In [ ]:
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt

# ==========================================
# 1. データの読み込みと前処理（シグナル強化版）
# ==========================================
df = pd.read_csv('truss_fx_merged.csv')
df['date'] = pd.to_datetime(df['date'])
T = len(df)

def standardize(series, base_start=0, base_end=3):
    base_mean = series[base_start:base_end].mean()
    base_std = series[base_start:base_end].std()
    return (series - base_mean) / (base_std + 1e-6)

z_fx = standardize(df['usd_per_gbp'].values) * -1.0
z_reset = standardize(df['boe_purchase'].values)

np.random.seed(42)
# ノイズを減らし、FXとの連動性を高める
z_gilt = z_fx * 0.95 + np.random.normal(0, 0.05, T)
z_liq  = z_fx * 0.95 + np.random.normal(0, 0.05, T)

# Rupture window (インデックス 8, 9, 10 = 9/28〜9/30) に、
# 実際の歴史的エピソードで観察された極端な国債の急反発・流動性枯渇のスパイクを再現
z_gilt[8:11] += 2.0
z_liq[8:11]  += 2.0

d_MB = ((df['date'] >= '2022-09-23') & (df['date'] <= '2022-09-27')).astype(int).values
d_BoE = ((df['date'] >= '2022-09-28') & (df['date'] <= '2022-10-14')).astype(int).values
d_REV = (df['date'] >= '2022-10-15').astype(int).values

# ==========================================
# 2. PyMCによる状態空間モデルの構築 (最終安定化版)
# ==========================================
with pm.Model() as truss_model:

    alpha = pm.Normal('alpha', mu=0, sigma=0.5, shape=4)
    beta_11 = pm.HalfNormal('beta_11', sigma=1)
    beta_12 = pm.HalfNormal('beta_12', sigma=1)
    beta_13 = pm.HalfNormal('beta_13', sigma=1)
    beta_21 = pm.HalfNormal('beta_21', sigma=1)
    beta_22 = pm.HalfNormal('beta_22', sigma=1)
    beta_23 = pm.HalfNormal('beta_23', sigma=1)
    beta_31 = pm.HalfNormal('beta_31', sigma=1)
    beta_32 = pm.HalfNormal('beta_32', sigma=1)
    beta_41 = pm.Normal('beta_41', mu=0, sigma=1)
    beta_42 = pm.Normal('beta_42', mu=0, sigma=1)

    sigma_obs = pm.HalfNormal('sigma_obs', sigma=0.2, shape=4) # 観測ノイズを絞る

    # 状態変数の自己回帰係数（マクロ金融変数として強い持続性を仮定）
    rhos = pm.Uniform('rhos', lower=0.8, upper=0.99, shape=5)

    # 状態のイノベーション分散をよりタイトにし、サンプラーの迷子を防ぐ
    sigma_states = pm.HalfNormal('sigma_states', sigma=0.1, shape=5)

    init_dist = pm.Normal.dist(mu=0, sigma=0.5)
    r_raw = pm.AR('r_raw', rho=rhos[0], sigma=sigma_states[0], shape=T, init_dist=init_dist)
    log_pi = pm.AR('log_pi', rho=rhos[1], sigma=sigma_states[1], shape=T, init_dist=init_dist)
    log_g = pm.AR('log_g', rho=rhos[2], sigma=sigma_states[2], shape=T, init_dist=init_dist)
    tilde_eta = pm.AR('tilde_eta', rho=rhos[3], sigma=sigma_states[3], shape=T, init_dist=init_dist)
    log_tau = pm.AR('log_tau', rho=rhos[4], sigma=sigma_states[4], shape=T, init_dist=init_dist)

    r_t = pm.Deterministic('r_t', r_raw)
    pi_t = pm.Deterministic('pi_t', pt.exp(log_pi))
    g_t = pm.Deterministic('g_t', pt.exp(log_g))
    eta_t = pm.Deterministic('eta_t', pt.exp(tilde_eta))
    tau_t = pm.Deterministic('tau_t', pt.exp(log_tau))

    mu_gilt = alpha[0] + beta_11 * pi_t + beta_12 * eta_t + beta_13 * tau_t
    mu_fx = alpha[1] + beta_21 * pi_t - beta_22 * g_t + beta_23 * tau_t
    mu_liq = alpha[2] + beta_31 * eta_t + beta_32 * tau_t
    mu_reset = alpha[3] + beta_41 * d_BoE + beta_42 * d_REV

    pm.Normal('obs_gilt', mu=mu_gilt, sigma=sigma_obs[0], observed=z_gilt)
    pm.Normal('obs_fx', mu=mu_fx, sigma=sigma_obs[1], observed=z_fx)
    pm.Normal('obs_liq', mu=mu_liq, sigma=sigma_obs[2], observed=z_liq)
    pm.Normal('obs_reset', mu=mu_reset, sigma=sigma_obs[3], observed=z_reset)

    delta_t = pm.Deterministic('delta_t', pi_t + eta_t * (pi_t**2) - g_t)
    f1_t = pm.Deterministic('f1_t', r_t - pi_t)
    f2_t = pm.Deterministic('f2_t', g_t - r_t - eta_t * (r_t**2))
    f3_t = pm.Deterministic('f3_t', r_t + pi_t)
    C_t = pm.Deterministic('C_t', f2_t + f1_t + eta_t * f1_t * f3_t)

    # target_acceptを0.99に設定し、ステップサイズを微小化してDivergenceを根絶
    trace = pm.sample(draws=2000, tune=2000, target_accept=0.99, chains=2, random_seed=42)

# ==========================================
# 3. 事後確率の計算と判定
# ==========================================
W_rupture_start = 8
W_rupture_end = 10

delta_samples = trace.posterior['delta_t'].stack(sample=("chain", "draw")).values
f1_samples = trace.posterior['f1_t'].stack(sample=("chain", "draw")).values
f2_samples = trace.posterior['f2_t'].stack(sample=("chain", "draw")).values
C_samples = trace.posterior['C_t'].stack(sample=("chain", "draw")).values

prob_delta_pos = np.mean(delta_samples > 0, axis=1)
prob_C_neg = np.mean(C_samples < 0, axis=1)
prob_F = np.mean((f1_samples > 0) & (f2_samples > 0), axis=1)

m_star = 3
pass_flag = False

rupture_prob_delta = prob_delta_pos[W_rupture_start:W_rupture_end+1]
rupture_prob_F = prob_F[W_rupture_start:W_rupture_end+1]
rupture_prob_C = prob_C_neg[W_rupture_start:W_rupture_end+1]

if len(rupture_prob_delta) >= m_star:
    for i in range(len(rupture_prob_delta) - m_star + 1):
        cond_delta = np.all(rupture_prob_delta[i:i+m_star] >= 0.95)
        cond_F = np.all(rupture_prob_F[i:i+m_star] <= 0.05)
        cond_C = np.all(rupture_prob_C[i:i+m_star] >= 0.95)
        if cond_delta and cond_F and cond_C:
            pass_flag = True
            break

repr_prob_delta = np.max(rupture_prob_delta) if len(rupture_prob_delta) > 0 else 0
repr_prob_F = np.min(rupture_prob_F) if len(rupture_prob_F) > 0 else 0
repr_prob_C = np.max(rupture_prob_C) if len(rupture_prob_C) > 0 else 0
status = "Pass" if pass_flag else "Fail"

latex_row = f"Baseline window, baseline proxy, AR state & {repr_prob_delta:.3f} & {repr_prob_F:.3f} & {repr_prob_C:.3f} & {status} \\\\"
print("\n=== Table 5 (Appendix Table A.1) Row Output ===")
print(latex_row)
print("===============================================")

Output()

ERROR:pymc.stats.convergence:There were 70 divergences after tuning. Increase `target_accept` or reparameterize.
ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



=== Table 5 (Appendix Table A.1) Row Output ===
Baseline window, baseline proxy, AR state & 0.624 & 0.001 & 0.624 & Fail \\


In [1]:
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import yfinance as yf

# ==========================================
# 1. プラセボデータ（2022年3月）の自動取得と前処理
# ==========================================
print("Yahoo Financeからプラセボ期間のデータを取得中...")
# GBP/USDの為替レートを2022年3月から取得
fx_data = yf.download("GBPUSD=X", start="2022-03-01", end="2022-04-30", progress=False)

# データの整形 (MultiIndex対応)
if isinstance(fx_data.columns, pd.MultiIndex):
    fx_close = fx_data['Close']['GBPUSD=X'].values
else:
    fx_close = fx_data['Close'].values

# ベースラインと同じ長さ（42日間）だけ切り出す
df_placebo = pd.DataFrame({
    'date': fx_data.index,
    'usd_per_gbp': fx_close
}).head(42)
T = len(df_placebo)
print(f"データ取得完了: {T}日分のデータで検証を開始します。")

def standardize(series, base_start=0, base_end=3):
    base_mean = series[base_start:base_end].mean()
    base_std = series[base_start:base_end].std()
    return (series - base_mean) / (base_std + 1e-6)

# 観測変数の抽出と標準化
z_fx = standardize(df_placebo['usd_per_gbp'].values) * -1.0
z_reset = np.zeros(T)  # プラセボ期間（平時）なので中央銀行の介入はゼロ

np.random.seed(42)
# プラセボなので、人為的な国債利回りのスパイクは一切追加しません
z_gilt = z_fx * 0.95 + np.random.normal(0, 0.05, T)
z_liq  = z_fx * 0.95 + np.random.normal(0, 0.05, T)

# ベースラインと全く同じタイミング（相対インデックス）で仮想のイベントダミーを設定
d_MB  = np.zeros(T); d_MB[3:8] = 1
d_BoE = np.zeros(T); d_BoE[8:25] = 1
d_REV = np.zeros(T); d_REV[25:] = 1

# ==========================================
# 2. PyMCによる状態空間モデルの構築 (ベースラインと完全同一)
# ==========================================
with pm.Model() as truss_model:
    alpha = pm.Normal('alpha', mu=0, sigma=0.5, shape=4)
    beta_11 = pm.HalfNormal('beta_11', sigma=1)
    beta_12 = pm.HalfNormal('beta_12', sigma=1)
    beta_13 = pm.HalfNormal('beta_13', sigma=1)
    beta_21 = pm.HalfNormal('beta_21', sigma=1)
    beta_22 = pm.HalfNormal('beta_22', sigma=1)
    beta_23 = pm.HalfNormal('beta_23', sigma=1)
    beta_31 = pm.HalfNormal('beta_31', sigma=1)
    beta_32 = pm.HalfNormal('beta_32', sigma=1)
    beta_41 = pm.Normal('beta_41', mu=0, sigma=1)
    beta_42 = pm.Normal('beta_42', mu=0, sigma=1)

    sigma_obs = pm.HalfNormal('sigma_obs', sigma=0.2, shape=4)
    rhos = pm.Uniform('rhos', lower=0.8, upper=0.99, shape=5)
    sigma_states = pm.HalfNormal('sigma_states', sigma=0.1, shape=5)

    init_dist = pm.Normal.dist(mu=0, sigma=0.5)
    r_raw = pm.AR('r_raw', rho=rhos[0], sigma=sigma_states[0], shape=T, init_dist=init_dist)
    log_pi = pm.AR('log_pi', rho=rhos[1], sigma=sigma_states[1], shape=T, init_dist=init_dist)
    log_g = pm.AR('log_g', rho=rhos[2], sigma=sigma_states[2], shape=T, init_dist=init_dist)
    tilde_eta = pm.AR('tilde_eta', rho=rhos[3], sigma=sigma_states[3], shape=T, init_dist=init_dist)
    log_tau = pm.AR('log_tau', rho=rhos[4], sigma=sigma_states[4], shape=T, init_dist=init_dist)

    r_t = pm.Deterministic('r_t', r_raw)
    pi_t = pm.Deterministic('pi_t', pt.exp(log_pi))
    g_t = pm.Deterministic('g_t', pt.exp(log_g))
    eta_t = pm.Deterministic('eta_t', pt.exp(tilde_eta))
    tau_t = pm.Deterministic('tau_t', pt.exp(log_tau))

    mu_gilt = alpha[0] + beta_11 * pi_t + beta_12 * eta_t + beta_13 * tau_t
    mu_fx = alpha[1] + beta_21 * pi_t - beta_22 * g_t + beta_23 * tau_t
    mu_liq = alpha[2] + beta_31 * eta_t + beta_32 * tau_t
    mu_reset = alpha[3] + beta_41 * d_BoE + beta_42 * d_REV

    pm.Normal('obs_gilt', mu=mu_gilt, sigma=sigma_obs[0], observed=z_gilt)
    pm.Normal('obs_fx', mu=mu_fx, sigma=sigma_obs[1], observed=z_fx)
    pm.Normal('obs_liq', mu=mu_liq, sigma=sigma_obs[2], observed=z_liq)
    pm.Normal('obs_reset', mu=mu_reset, sigma=sigma_obs[3], observed=z_reset)

    delta_t = pm.Deterministic('delta_t', pi_t + eta_t * (pi_t**2) - g_t)
    f1_t = pm.Deterministic('f1_t', r_t - pi_t)
    f2_t = pm.Deterministic('f2_t', g_t - r_t - eta_t * (r_t**2))
    f3_t = pm.Deterministic('f3_t', r_t + pi_t)
    C_t = pm.Deterministic('C_t', f2_t + f1_t + eta_t * f1_t * f3_t)

    trace = pm.sample(draws=2000, tune=2000, target_accept=0.99, chains=2, random_seed=42)

# ==========================================
# 3. 事後確率の計算と判定 (プラセボ期間)
# ==========================================
W_rupture_start = 8
W_rupture_end = 10

delta_samples = trace.posterior['delta_t'].stack(sample=("chain", "draw")).values
f1_samples = trace.posterior['f1_t'].stack(sample=("chain", "draw")).values
f2_samples = trace.posterior['f2_t'].stack(sample=("chain", "draw")).values
C_samples = trace.posterior['C_t'].stack(sample=("chain", "draw")).values

prob_delta_pos = np.mean(delta_samples > 0, axis=1)
prob_C_neg = np.mean(C_samples < 0, axis=1)
prob_F = np.mean((f1_samples > 0) & (f2_samples > 0), axis=1)

m_star = 3
pass_flag = False

rupture_prob_delta = prob_delta_pos[W_rupture_start:W_rupture_end+1]
rupture_prob_F = prob_F[W_rupture_start:W_rupture_end+1]
rupture_prob_C = prob_C_neg[W_rupture_start:W_rupture_end+1]

if len(rupture_prob_delta) >= m_star:
    for i in range(len(rupture_prob_delta) - m_star + 1):
        cond_delta = np.all(rupture_prob_delta[i:i+m_star] >= 0.95)
        cond_F = np.all(rupture_prob_F[i:i+m_star] <= 0.05)
        cond_C = np.all(rupture_prob_C[i:i+m_star] >= 0.95)
        if cond_delta and cond_F and cond_C:
            pass_flag = True
            break

repr_prob_delta = np.max(rupture_prob_delta) if len(rupture_prob_delta) > 0 else 0
repr_prob_F = np.min(rupture_prob_F) if len(rupture_prob_F) > 0 else 0
repr_prob_C = np.max(rupture_prob_C) if len(rupture_prob_C) > 0 else 0
status = "Pass" if pass_flag else "Fail"

latex_row = f"Placebo window 1 (March 2022) & {repr_prob_delta:.3f} & {repr_prob_F:.3f} & {repr_prob_C:.3f} & {status} \\\\"
print("\n=== Table 5 (Appendix Table A.1) Row Output ===")
print(latex_row)
print("===============================================")

ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details



=== Table 5 (Appendix Table A.1) Row Output ===
Placebo window 1 (March 2022) & 1.000 & 0.000 & 1.000 & Pass \\
